In [43]:
# --- Cell 2: Imports ---
from rdflib import Graph

# Notice we use 'src.filename' now!
from schema_definition import build_schema
from integrate_rQTLs import add_rqtl_to_graph
from integrate_gwas_catalog import enrich_graph_with_gwas
from integrate_open_targets import enrich_graph_with_opentargets
#from integrate_reactome import enrich_graph_with_reactome
from schema_definition import *
from execute_sparql import execute_sparql
from graphdb_engine import query_graphdb
from integrate_string import enrich_graph_with_string
from integrate_encode import enrich_graph_with_encode

# Integrate rQTLs

In [31]:
# 1. Initialize Graph
import random
from tqdm import tqdm
import glob
g = Graph()
g = build_schema(g)
number_of_rqtls= 100
rqtl_files = glob.glob("../data/json_files/*.json")
rqtl_files= random.sample(rqtl_files, number_of_rqtls)
# 2. Define Schema & Ontologies
rqtl_json_path = "../data/json_files/GCST90200330_GCST90200020.json"

# 3. Ingest Local JSON Data
print("\n[STEP 1] Ingesting local rQTL data...")
#add_rqtl_to_graph(rqtl_json_path, g)
for rqtl_file in tqdm(rqtl_files,total=number_of_rqtls):
    g = add_rqtl_to_graph(rqtl_file, g)





[STEP 1] Ingesting local rQTL data...


100%|██████████| 100/100 [00:05<00:00, 18.77it/s]


# Integrate GWAS Catalog

In [32]:
import numpy as np
# 4. Enrich with External APIs
number_of_snps= len(list(g.subjects(RDF.type, MTR.SNP)))

print("\n[STEP 2] Enriching with External Databases...")
g = enrich_graph_with_gwas(g,max_snps=number_of_snps)



[STEP 2] Enriching with External Databases...

--- Starting GWAS Catalog Integration (v2 API) ---
[1/79] Fetching v2 associations for rs662138...
   -> Found 10 significant associations!
[2/79] Fetching v2 associations for rs603424...
   -> Found 10 significant associations!
[3/79] Fetching v2 associations for rs174546...
   -> Found 10 significant associations!
[4/79] Fetching v2 associations for rs174564...
   -> Found 10 significant associations!
[5/79] Fetching v2 associations for rs174599...
   -> Found 10 significant associations!
[6/79] Fetching v2 associations for rs34400381...
   -> Found 10 significant associations!
[7/79] Fetching v2 associations for rs148982377...
   -> Found 10 significant associations!
[8/79] Fetching v2 associations for rs2472297...
   -> Found 10 significant associations!
[9/79] Fetching v2 associations for rs11211406...
   -> Found 2 significant associations!
[10/79] Fetching v2 associations for rs1165209...
   -> Found 6 significant associations!
[11

# Integrate Open Targets

In [33]:
number_of_genes=len(list(g.subjects(RDF.type, MTR.Gene)))


In [34]:

g = enrich_graph_with_opentargets(g, max_genes=number_of_genes)



--- Starting Open Targets Integration ---
Found 187 causal Genes in the local graph.
[1/187] Fetching OT data for SOD2...
   -> Found 4 safety liabilities!
   -> Found 10 top associated diseases!
[2/187] Fetching OT data for MRPL18...
   -> Found 10 top associated diseases!
[3/187] Fetching OT data for ABCC2...
   -> Found 20 safety liabilities!
   -> Found 10 top associated diseases!
[4/187] Fetching OT data for SCD...
   -> Found 4 safety liabilities!
   -> Found 10 top associated diseases!
[5/187] Fetching OT data for DNMBP...
   -> Found 10 top associated diseases!
[6/187] Fetching OT data for BLOC1S2...
   -> Found 10 top associated diseases!
[7/187] Fetching OT data for TMEM258...
   -> Found 10 top associated diseases!
[8/187] Fetching OT data for FADS1...
   -> Found 10 top associated diseases!
[9/187] Fetching OT data for FADS2...
   -> Found 10 top associated diseases!
[10/187] Fetching OT data for FADS3...
   -> Found 10 top associated diseases!
[11/187] Fetching OT data fo

# Integrate STRING database

In [35]:
g=enrich_graph_with_string(g)


--- Starting STRING Database Integration ---
Found 187 genes. Fetching interaction network...
Successfully retrieved 200 interactions from STRING.
--- STRING Integration Complete ---



# Integrate ENCODE database

In [46]:
g=enrich_graph_with_encode(g)



2026-03-07 14:43:38,217 [INFO] --- Starting ENCODE Epigenetic Integration (via Ensembl) ---
2026-03-07 14:43:38,218 [INFO] [1/79] Checking ENCODE regulatory overlaps for rs662138...
2026-03-07 14:43:38,922 [INFO]    -> No overlapping epigenetic features found.
2026-03-07 14:43:39,023 [INFO] [2/79] Checking ENCODE regulatory overlaps for rs603424...
2026-03-07 14:43:39,747 [INFO]    -> No overlapping epigenetic features found.
2026-03-07 14:43:39,848 [INFO] [3/79] Checking ENCODE regulatory overlaps for rs174546...
2026-03-07 14:43:40,721 [INFO]    -> No overlapping epigenetic features found.
2026-03-07 14:43:40,822 [INFO] [4/79] Checking ENCODE regulatory overlaps for rs174564...
2026-03-07 14:43:41,607 [INFO]    -> Found 1 ENCODE regulatory features!
2026-03-07 14:43:41,758 [INFO] [5/79] Checking ENCODE regulatory overlaps for rs174599...
2026-03-07 14:43:42,574 [INFO]    -> No overlapping epigenetic features found.
2026-03-07 14:43:42,675 [INFO] [6/79] Checking ENCODE regulatory over

In [52]:
import pandas as pd
df_report=pd.read_csv("../output/encode_mapping_report.csv")

In [55]:
df_report[df_report.Status=="Mapped"]

,SNP,Status,Features_Found
3,rs174564,Mapped,1
16,rs73355734,Mapped,1
18,rs2062541,Mapped,1
28,rs61797339,Mapped,1
41,rs1767447,Mapped,1
52,rs924135,Mapped,1
58,rs174560,Mapped,1


In [ ]:
# Integrate STRING database


In [39]:
from graphdb_engine import query_graphdb

# The optimized query from earlier
optimized_gwas_query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX oban: <http://purl.org/oban/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?snp ?ratio_name ?trait_name ?gwas_p_value ?gwas_beta
WHERE {
  # 1. Get the rQTL Associations
  ?ratio a mtr:MetaboliteRatio ;
         rdfs:label ?ratio_name .

  ?rqtl_assoc a oban:association ;
              oban:has_object ?ratio ;
              oban:has_subject ?snp .

  # 2. Get the GWAS Associations for those exact SNPs
  ?gwas_assoc a biolink:Association ;
              biolink:has_subject ?snp ;
              biolink:has_object ?trait ;
              mtr:p_value ?gwas_p_value .

  ?trait rdfs:label ?trait_name .

  # 3. Grab the beta if it exists
  OPTIONAL { ?gwas_assoc mtr:beta ?gwas_beta . }

  # 4. Filter for genome-wide significance
  FILTER (?gwas_p_value < 0.00000005)
}
ORDER BY ?gwas_p_value
LIMIT 10

"""

# Execute the query against GraphDB!
# Note: Ensure the 'repo_name' matches what you named it in the GraphDB Workbench
df_results = query_graphdb(optimized_gwas_query, repo_name="mtrKG")

# Display the results
display(df_results)

Sending query to GraphDB (mtrKG)...
Success! Retrieved 10 rows.


,snp,ratio_name,trait_name,gwas_p_value,gwas_beta
0,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.35285
1,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.352894
2,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.352233
3,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.351561
4,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.350063
5,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.348978
6,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,2e-286,-0.245665
7,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,5.0000000000000003e-284,-0.35285
8,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,5.0000000000000003e-284,-0.352894
9,http://identifiers.org/dbsnp/rs102275,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero...",lysophosphatidylethanolamine measurement,5.0000000000000003e-284,-0.352233


In [15]:
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX oban: <http://purl.org/oban/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?snp ?ratio_name ?trait_name ?gwas_p_value ?gwas_beta
WHERE {
  ?rqtl_assoc a oban:association ;
              oban:has_subject ?snp ;
              oban:has_object ?ratio .
  ?ratio rdfs:label ?ratio_name .

  ?gwas_assoc a biolink:Association ;
              biolink:has_subject ?snp ;
              biolink:has_object ?trait ;
              mtr:p_value ?gwas_p_value .
  ?trait rdfs:label ?trait_name .

  OPTIONAL { ?gwas_assoc mtr:beta ?gwas_beta . }

  FILTER (?gwas_p_value < 0.00000005)
}
ORDER BY ?gwas_p_value
LIMIT 10
"""

In [16]:
print(g.serialize(format="turtle"))

@prefix biolink: <https://w3id.org/biolink/vocab/> .
@prefix chebi: <http://purl.obolibrary.org/obo/CHEBI_> .
@prefix dbsnp: <http://identifiers.org/dbsnp/> .
@prefix efo: <http://www.ebi.ac.uk/efo/> .
@prefix hmdb: <http://identifiers.org/hmdb/> .
@prefix mtr: <https://metabolite-ratio-app.unil.ch/rqtl/> .
@prefix ns1: <http://purl.org/oban/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix pubmed: <https://pubmed.ncbi.nlm.nih.gov/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sio: <http://semanticscience.org/resource/> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ns1:has_object owl:equivalentProperty biolink:has_object .

ns1:has_subject owl:equivalentProperty biolink:has_subject .

<https://metabolite-ratio-app.unil.ch/ontology> a owl:Ontology ;
    rdfs:label "rQTL Metabolic Ratio Ontology" ;
    rdfs:comment "A schema defining genetically regul

<font size="4"> "What are the most statistically significant genetic associations in the GWAS Catalog currently stored in our graph, and what are their specific effect sizes, risk alleles, and original publication IDs?"
</font>

In [38]:
df_results = query_graphdb("../analysis/002_query.rq", repo_name="mtrKG")

# Display the results
df_results

Reading SPARQL query from file: ../analysis/002_query.rq
Sending query to GraphDB (mtrKG)...
Success! Retrieved 118 rows.


,snp_id,strongest_gwas_p_value,associated_clinical_traits,driven_metabolic_ratios
0,rs102275,4e-264,coronary artery calcification | lysophosphatid...,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero..."
1,rs1047891,0.0,glycine measurement | gamma-glutamylglycine me...,choline to isovalerylglycine | n-palmitoylglyc...
2,rs1260326,0.0,triglyceride measurement | triglycerides in me...,"sphingomyelin (d17:1/16:0, d18:1/15:0, d16:1/1..."
3,rs174560,4e-59,1-stearoyl-2-arachidonoyl-GPI (18:0/20:4) meas...,dihomo-linolenate (20:3n3 or n6) to arachidono...
4,rs4814176,3e-107,level of ceramide | sphingomyelin measurement ...,lignoceroyl sphingomyelin (d18:1/24:0) to sphi...
...,...,...,...,...
113,rs1767447,2e-11,heel bone mineral density,phosphoethanolamine to 1-stearoyl-2-oleoyl-gps...
114,rs12095130,3e-10,X-21607 measurement,x-16397 to x-17335
115,rs7573275,2e-09,self reported educational attainment | N-delta...,"n2-acetyl,n6,n6-dimethyllysine to x-12112 | 2-..."
116,rs34048349,2e-09,body height,"5alpha-androstan-3beta,17beta-diol monosulfate..."


<font size="4"> Find me a severe clinical disease from the GWAS Catalog that is driven by a specific genetic variant. Then, show me the metabolic ratio disrupted by that variant, the causal enzyme responsible for the disruption, and finally, check if there is an FDA-approved drug (Phase 3 or 4) that already targets that enzyme."
</font>

In [40]:
df_results = query_graphdb("../analysis/003_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/003_query.rq
Sending query to GraphDB (mtrKG)...
Success! Retrieved 20 rows.


,disease_name,ratio_name,causal_gene,drug_name,max_phase
0,16a-hydroxy DHEA 3-sulfate measurement,"androstenediol (3beta,17beta) disulfate (2) to...",CYP3A5,RITONAVIR,4
1,16a-hydroxy DHEA 3-sulfate measurement,"5alpha-androstan-3beta,17beta-diol monosulfate...",CYP3A5,RITONAVIR,4
2,16a-hydroxy DHEA 3-sulfate measurement,"androstenediol (3beta,17beta) disulfate (2) to...",CYP3A7,RITONAVIR,4
3,16a-hydroxy DHEA 3-sulfate measurement,"5alpha-androstan-3beta,17beta-diol monosulfate...",CYP3A7,RITONAVIR,4
4,16a-hydroxy DHEA 3-sulfate measurement,"androstenediol (3beta,17beta) disulfate (2) to...",CYP3A5,COBICISTAT,4
5,16a-hydroxy DHEA 3-sulfate measurement,"5alpha-androstan-3beta,17beta-diol monosulfate...",CYP3A5,COBICISTAT,4
6,16a-hydroxy DHEA 3-sulfate measurement,"androstenediol (3beta,17beta) disulfate (2) to...",CYP3A7,COBICISTAT,4
7,16a-hydroxy DHEA 3-sulfate measurement,"5alpha-androstan-3beta,17beta-diol monosulfate...",CYP3A7,COBICISTAT,4
8,"5alpha-androstan-3alpha,17beta-diol monosulfat...","androstenediol (3beta,17beta) disulfate (2) to...",CYP3A5,RITONAVIR,4
9,"5alpha-androstan-3alpha,17beta-diol monosulfat...","5alpha-androstan-3beta,17beta-diol monosulfate...",CYP3A5,RITONAVIR,4


<font size="4"> We found a gene that causally drives a metabolic imbalance linked to a disease. However, what if that specific gene is considered 'undruggable' (no known drugs)? Can the Knowledge Graph find an FDA-approved drug that targets a different protein, as long as that protein physically binds to our undruggable causal gene?"
</font>

In [41]:
df_results = query_graphdb("../analysis/004_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/004_query.rq
Sending query to GraphDB (mtrKG)...
Success! Retrieved 20 rows.


,ratio_name,causal_gene_name,interacting_gene_name,interaction_score,drug_name,clinical_phase
0,"androstenediol (3beta,17beta) disulfate (2) to...",CYP3A5,CYP3A7,0.968,RITONAVIR,4
1,"5alpha-androstan-3beta,17beta-diol monosulfate...",CYP3A5,CYP3A7,0.968,RITONAVIR,4
2,"androstenediol (3beta,17beta) disulfate (2) to...",CYP3A5,CYP3A7,0.968,COBICISTAT,4
3,"5alpha-androstan-3beta,17beta-diol monosulfate...",CYP3A5,CYP3A7,0.968,COBICISTAT,4
4,3-hydroxysebacate to 2-butenoylglycine,CYP4A11,CYP3A7,0.948,RITONAVIR,4
5,3-hydroxysebacate to x-14939,CYP4A11,CYP3A7,0.948,RITONAVIR,4
6,2-hydroxysebacate to x-18913,CYP4A11,CYP3A7,0.948,RITONAVIR,4
7,x-16580 to x-12839,CYP4A11,CYP3A7,0.948,RITONAVIR,4
8,x-16397 to x-17335,CYP4A11,CYP3A7,0.948,RITONAVIR,4
9,pregnenolone sulfate to 21-hydroxypregnenolone...,CYP4A11,CYP3A7,0.948,RITONAVIR,4


<font size="4"> ENCODE for epigenetics "
</font>

In [56]:
df_results = query_graphdb("../analysis/005_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/005_query.rq
Sending query to GraphDB (mtrKG)...
Success! Retrieved 7 rows.


,snp_id,regulatory_element
0,rs174560,enhancer (ENSR11_C85LJ)
1,rs174564,enhancer (ENSR11_9WJJ2)
2,rs1767447,enhancer (ENSR1_C45RM)
3,rs2062541,enhancer (ENSR16_98CSB)
4,rs61797339,promoter (ENSR1_9353HX)
5,rs73355734,enhancer (ENSR17_935633)
6,rs924135,enhancer (ENSR16_98CR7)


In [30]:
execute_sparql(g,optimized_gwas_query)

Executing SPARQL query...


KeyboardInterrupt: 

In [37]:
execute_sparql(g,optimized_gwas_query)

Executing SPARQL query...


KeyboardInterrupt: 

In [47]:
g.serialize(destination="../output/mtrKG.ttl", format="turtle")

<Graph identifier=N49979e76e56647f580b9ce679322d00c (<class 'rdflib.graph.Graph'>)>

In [ ]:
g = enrich_graph_with_reactome(g)

# 5. NetworkX Graph Analytics
print("\n[STEP 3] Running Graph Analytics...")
g = calculate_reaction_distances(g)

# 6. SHACL Validation
print("\n[STEP 4] Validating Graph structure...")
validate_graph_with_shacl(g)

# 7. Save Output
output_path = "output/final_knowledge_graph.ttl"
g.serialize(destination=output_path, format="turtle")
print(f"\n[SUCCESS] Knowledge Graph saved to {output_path} with {len(g)} triples!")